In [3]:
!#!pip install langchain langchain-openai langchain-community chromadb
! pip install -U langchain langchain-community langchain-openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [4]:
import chromadb
from openai import OpenAI

In [13]:
#!export OPENAI_API_KEY=""
!export OPENAI_API_KEY=""


In [15]:
OPENAI_API_KEY=""


In [16]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)

In [17]:
def create_rag():
    """Create and return RAG components."""
    #client = OpenAI()
    client = OpenAI(
    api_key=OPENAI_API_KEY
    )
    db = chromadb.Client()
    collection = db.create_collection("docs")
    return client, collection


def add_documents(collection, documents: list[str]):
    """Add documents to the vector store."""
    collection.add(
        documents=documents,
        ids=[f"doc_{i}" for i in range(len(documents))],
    )


def query(client, collection, question: str, n_results: int = 2) -> str:
    """
    RAG query: retrieve relevant docs and generate answer.

    Args:
        client: OpenAI client
        collection: ChromaDB collection
        question: User's question
        n_results: Number of documents to retrieve

    Returns:
        Generated answer based on retrieved context
    """
    # Retrieve relevant documents
    results = collection.query(query_texts=[question], n_results=n_results)
    context = "\n".join(results["documents"][0])

    # Generate answer with context
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Answer based on the provided context. Be concise.",
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
    )
    return response.choices[0].message.content


def main():
    # Sample knowledge base
    documents = [
        "Python was created by Guido van Rossum and first released in 1991.",
        "Python emphasizes code readability with significant indentation.",
        "The Zen of Python includes principles like 'Simple is better than complex'.",
        "Python supports multiple paradigms: procedural, OOP, and functional.",
        "Popular Python frameworks include Django, Flask, and FastAPI.",
    ]

    # Initialize RAG
    client, collection = create_rag()
    add_documents(collection, documents)

    # Example queries
    questions = [
        "Who created Python?",
        "What are Python's design principles?",
    ]

    for q in questions:
        print(f"Q: {q}")
        print(f"A: {query(client, collection, q)}\n")


if __name__ == "__main__":
    main()

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 39.7MiB/s]


Q: Who created Python?
A: Python was created by Guido van Rossum.

Q: What are Python's design principles?
A: Python's design principles include code readability, simplicity, and explicitness. It often promotes the idea that "there should be one—and preferably only one—obvious way to do it." Additionally, Python encourages the use of significant indentation to enhance clarity and structure in code.

